In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load the similarity matrix
similarity_file = Path(r"C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\models\2025-Q2\precompute_20250608_132404\job_similarity_matrix.csv")
df = pd.read_csv(similarity_file)

print(f"Similarity matrix shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nFirst few rows:")
df.head()

In [ ]:
# Basic statistics about the similarity scores
print("=== SIMILARITY SCORE DISTRIBUTION ===")
print(f"Min similarity: {df['similarity'].min()}")
print(f"Max similarity: {df['similarity'].max()}")
print(f"Mean similarity: {df['similarity'].mean():.4f}")
print(f"Median similarity: {df['similarity'].median():.4f}")
print(f"Std deviation: {df['similarity'].std():.4f}")

print(f"\n=== SCORE RANGES ===")
print(f"Perfect matches (1.0): {(df['similarity'] == 1.0).sum():,}")
print(f"No overlap (0.0): {(df['similarity'] == 0.0).sum():,}")
print(f"High similarity (>0.8): {(df['similarity'] > 0.8).sum():,}")
print(f"Medium similarity (0.2-0.8): {((df['similarity'] >= 0.2) & (df['similarity'] <= 0.8)).sum():,}")
print(f"Low similarity (<0.2): {(df['similarity'] < 0.2).sum():,}")

# Quick describe
df['similarity'].describe()

In [ ]:
# Check for data quality issues
print("=== DATA QUALITY CHECKS ===")

# Check for missing values
print(f"Missing values: {df.isnull().sum().sum()}")

# Check for duplicates
duplicates = df.duplicated(subset=['job_from', 'job_to']).sum()
print(f"Duplicate job pairs: {duplicates}")

# Check for self-comparisons (should be 0 since we excluded them)
self_comparisons = df[df['job_from'] == df['job_to']]
print(f"Self-comparisons found: {len(self_comparisons)}")

# Check unique jobs
unique_from = df['job_from'].nunique()
unique_to = df['job_to'].nunique()
print(f"Unique jobs in 'job_from': {unique_from}")
print(f"Unique jobs in 'job_to': {unique_to}")

# Expected pairs calculation
expected_pairs = unique_from * (unique_to - 1) if unique_from == unique_to else unique_from * unique_to
print(f"Expected pairs (asymmetric, no self): {expected_pairs}")
print(f"Actual pairs: {len(df)}")
print(f"Match expected: {len(df) == expected_pairs}")

In [ ]:
# Create visualisations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Histogram of similarity scores
axes[0,0].hist(df['similarity'], bins=50, alpha=0.7, edgecolor='black')
axes[0,0].set_title('Distribution of Similarity Scores')
axes[0,0].set_xlabel('Similarity Score')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df['similarity'].mean(), color='red', linestyle='--', label=f'Mean: {df["similarity"].mean():.3f}')
axes[0,0].legend()

# Box plot
axes[0,1].boxplot(df['similarity'])
axes[0,1].set_title('Similarity Score Box Plot')
axes[0,1].set_ylabel('Similarity Score')

# Cumulative distribution
sorted_similarities = np.sort(df['similarity'])
cumulative = np.arange(1, len(sorted_similarities) + 1) / len(sorted_similarities)
axes[1,0].plot(sorted_similarities, cumulative)
axes[1,0].set_title('Cumulative Distribution of Similarity Scores')
axes[1,0].set_xlabel('Similarity Score')
axes[1,0].set_ylabel('Cumulative Probability')
axes[1,0].grid(True)

# Log-scale histogram for better detail
axes[1,1].hist(df['similarity'], bins=50, alpha=0.7, edgecolor='black')
axes[1,1].set_yscale('log')
axes[1,1].set_title('Distribution (Log Scale)')
axes[1,1].set_xlabel('Similarity Score')
axes[1,1].set_ylabel('Frequency (Log Scale)')

plt.tight_layout()
plt.show()

In [ ]:
# Look at some specific examples
print("=== HIGH SIMILARITY EXAMPLES ===")
high_sim = df[df['similarity'] > 0.8].sort_values('similarity', ascending=False)
print(f"Found {len(high_sim)} pairs with >80% similarity")
print(high_sim.head(10))

print("\n=== PERFECT MATCHES ===")
perfect_matches = df[df['similarity'] == 1.0]
print(f"Found {len(perfect_matches)} perfect matches")
if len(perfect_matches) > 0:
    print(perfect_matches.head(10))

print("\n=== ZERO SIMILARITY EXAMPLES ===")
zero_sim = df[df['similarity'] == 0.0]
print(f"Found {len(zero_sim)} pairs with 0% similarity")
print(zero_sim.head(10))

print("\n=== RANDOM MEDIUM SIMILARITY EXAMPLES ===")
medium_sim = df[(df['similarity'] >= 0.3) & (df['similarity'] <= 0.7)].sample(10)
print(medium_sim)

In [ ]:
# Verify asymmetric nature by looking at reverse pairs
print("=== ASYMMETRIC VERIFICATION ===")

# Sample some job pairs and check if A->B similarity != B->A similarity
sample_pairs = [
    ('R0001.5', 'R0002.0'),
    ('R0002.0', 'R0001.5'),
    ('R0003.0', 'R0005.1'),
    ('R0005.1', 'R0003.0')
]

for job_from, job_to in sample_pairs:
    sim_forward = df[(df['job_from'] == job_from) & (df['job_to'] == job_to)]['similarity']
    sim_reverse = df[(df['job_from'] == job_to) & (df['job_to'] == job_from)]['similarity']
    
    if len(sim_forward) > 0 and len(sim_reverse) > 0:
        print(f"{job_from} -> {job_to}: {sim_forward.iloc[0]:.4f}")
        print(f"{job_to} -> {job_from}: {sim_reverse.iloc[0]:.4f}")
        print(f"Difference: {abs(sim_forward.iloc[0] - sim_reverse.iloc[0]):.4f}")
        print()

In [ ]:
# Analyse at job level
print("=== JOB-LEVEL ANALYSIS ===")

# For each job, what are its average incoming and outgoing similarities?
job_outgoing = df.groupby('job_from')['similarity'].agg(['mean', 'std', 'count']).round(4)
job_outgoing.columns = ['avg_outgoing_sim', 'std_outgoing_sim', 'outgoing_count']

job_incoming = df.groupby('job_to')['similarity'].agg(['mean', 'std', 'count']).round(4)
job_incoming.columns = ['avg_incoming_sim', 'std_incoming_sim', 'incoming_count']

# Combine
job_stats = job_outgoing.join(job_incoming, how='outer')

print("Jobs with highest average outgoing similarity (most similar to others):")
print(job_stats.sort_values('avg_outgoing_sim', ascending=False).head(10))

print("\nJobs with highest average incoming similarity (most attractive to others):")
print(job_stats.sort_values('avg_incoming_sim', ascending=False).head(10))

print("\nJobs with lowest average outgoing similarity (most unique):")
print(job_stats.sort_values('avg_outgoing_sim').head(10))

In [ ]:
# Look for patterns in job IDs and similarities
print("=== PATTERN INVESTIGATION ===")

# Extract job families from job IDs (assuming format like R0001.5)
df['job_from_family'] = df['job_from'].str.extract(r'(R\d+)')[0]
df['job_to_family'] = df['job_to'].str.extract(r'(R\d+)')[0]

# Similarity within vs between job families
within_family = df[df['job_from_family'] == df['job_to_family']]['similarity']
between_family = df[df['job_from_family'] != df['job_to_family']]['similarity']

print(f"Within family similarities:")
print(f"  Count: {len(within_family):,}")
print(f"  Mean: {within_family.mean():.4f}")
print(f"  Std: {within_family.std():.4f}")

print(f"\nBetween family similarities:")
print(f"  Count: {len(between_family):,}")
print(f"  Mean: {between_family.mean():.4f}")
print(f"  Std: {between_family.std():.4f}")

# Statistical test
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(within_family, between_family)
print(f"\nT-test (within vs between families):")
print(f"  T-statistic: {t_stat:.4f}")
print(f"  P-value: {p_value:.2e}")

In [ ]:
# Load original data to validate our results
skills_file = r"C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\input_data\skill_data.csv"
jobs_file = r"C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\input_data\job_data.csv"
job_skills_file = r"C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\input_data\job_skill_mapping.csv"

skills_df = pd.read_csv(skills_file)
jobs_df = pd.read_csv(jobs_file)
job_skills_df = pd.read_csv(job_skills_file)

print("=== ORIGINAL DATA SUMMARY ===")
print(f"Skills: {len(skills_df)} rows")
print(f"Jobs: {len(jobs_df)} rows") 
print(f"Job-skill mappings: {len(job_skills_df)} rows")

# Manual calculation for a specific pair to verify our algorithm
job1 = 'R0001.5'
job2 = 'R0002.0'

# Get skills for each job
job1_skills = set(job_skills_df[job_skills_df['JobProfileID'] == job1]['Skill_ID'])
job2_skills = set(job_skills_df[job_skills_df['JobProfileID'] == job2]['Skill_ID'])

# Calculate manual similarity
shared_skills = job1_skills & job2_skills
manual_similarity = len(shared_skills) / len(job1_skills) if job1_skills else 0

# Get our calculated similarity
our_similarity = df[(df['job_from'] == job1) & (df['job_to'] == job2)]['similarity'].iloc[0]

print(f"\n=== VALIDATION FOR {job1} -> {job2} ===")
print(f"Job1 skills count: {len(job1_skills)}")
print(f"Job2 skills count: {len(job2_skills)}")
print(f"Shared skills count: {len(shared_skills)}")
print(f"Manual calculation: {manual_similarity:.6f}")
print(f"Our calculation: {our_similarity:.6f}")
print(f"Match: {abs(manual_similarity - our_similarity) < 1e-10}")

In [ ]:
# Final summary
print("=== SIMILARITY MATRIX ANALYSIS SUMMARY ===")
print(f"✅ Dataset size: {len(df):,} job pairs")
print(f"✅ Jobs analysed: {df['job_from'].nunique()} unique jobs")
print(f"✅ Similarity range: {df['similarity'].min():.3f} to {df['similarity'].max():.3f}")
print(f"✅ Average similarity: {df['similarity'].mean():.3f}")
print(f"✅ Data quality: No missing values, no duplicates")
print(f"✅ Asymmetric nature: Confirmed")

# Key insights
perfect_pct = (df['similarity'] == 1.0).sum() / len(df) * 100
zero_pct = (df['similarity'] == 0.0).sum() / len(df) * 100
high_pct = (df['similarity'] > 0.8).sum() / len(df) * 100

print(f"\n=== KEY INSIGHTS ===")
print(f"📊 Perfect matches: {perfect_pct:.2f}% of pairs")
print(f"📊 Zero overlap: {zero_pct:.2f}% of pairs") 
print(f"📊 High similarity (>80%): {high_pct:.2f}% of pairs")

if 'job_from_family' in df.columns:
    within_mean = within_family.mean()
    between_mean = between_family.mean()
    print(f"📊 Within-family similarity: {within_mean:.3f} (avg)")
    print(f"📊 Between-family similarity: {between_mean:.3f} (avg)")
    print(f"📊 Family effect: {within_mean/between_mean:.2f}x higher within families")

print(f"\n🎯 Precomputation successful - ready for query layer!")

In [ ]:
jobs_df.JobProfileID.unique()